In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv("Sensors_Data.csv")
df.head()

,Journal Link,TiC Mxene sensor,Functional Group,Response Time (in min),Current density (in μA/cm2),Detection Range (in M),Compound detected,EN of functional group,Atomic polarizability of functional group (in Å³),Mobility of functional group (in ×10⁻⁸ m²/V·s),Covalent Radii of functional group (in pm),Chemical formula of compound detected,N/O ratio,N/C ratio,C/H ratio,C/O ratio
0,https://www.sciencedirect.com/science/article/...,Graphene/TiC,–O,9.0,1480.000,5.55556*10-17 - 1.11111*10-8,Carcinoembryonic antigen(CEA),3.44,0.785,20.64,66.0,C3394H5266N930O1076S15,0.8643,0.2740,0.6445,3.1542
1,https://www.sciencedirect.com/science/article/...,Graphene/TiC,–NH2,9.0,1480.000,5.55556*10-17 - 1.11111*10-8,Carcinoembryonic antigen(CEA),3.04,1.096,7.62,71.0,C3394H5266N930O1076S15,0.8643,0.2740,0.6445,3.1542
2,https://www.sciencedirect.com/science/article/...,Graphene/TiC,–F,9.0,1480.000,5.55556*10-17 - 1.11111*10-8,Carcinoembryonic antigen(CEA),3.98,0.554,5.74,57.0,C3394H5266N930O1076S15,0.8643,0.2740,0.6445,3.1542
3,https://www.nature.com/articles/s41598-025-267...,PANI/TiC/Au,–NH–,NaN,452.707,7.03482*10-17 - 7.03482*10-14,Prostate-specific antigen (PSA),3.04,1.096,7.62,71.0,C1292H2020N352O361S15,0.9750,0.2724,0.6396,3.5789
4,https://www.mdpi.com/2079-6374/14/6/261,GO/TiC/CNT,–O,NaN,1775.700,10-6 - 9770*10-6,Hydrogen Peroxide (H2O2),3.44,0.785,20.64,66.0,H2O2,0.0000,NaN,0.0000,0.0000


In [3]:
det_col = "Detection Range (in M)"

# Split safely (handles NaN automatically)
ranges = df[det_col].astype(str).str.split(" - ", expand=True)

# If split fails, fill with NaN
ranges = ranges.reindex(columns=[0,1])

# Convert scientific format properly
min_vals = (ranges[0]
            .str.replace("*10-", "e-", regex=False)
            .str.replace("*10", "e", regex=False)
            .str.replace("10-", "1e-", regex=False)
           )

max_vals = (ranges[1]
            .str.replace("*10-", "e-", regex=False)
            .str.replace("*10", "e", regex=False)
            .str.replace("10-", "1e-", regex=False)
           )

# Convert to numeric safely (invalid values → NaN)
min_vals = pd.to_numeric(min_vals, errors='coerce')
max_vals = pd.to_numeric(max_vals, errors='coerce')

# Compute log detection range safely
log_range = np.log10(max_vals / min_vals)
df["Log_Detection_Range"] = log_range

In [4]:
# Remove column if already present
df.drop(columns=["Log_Detection_Range"], errors="ignore", inplace=True)

# Find position of Detection Range column
col_index = df.columns.get_loc("Detection Range (in M)")

# Insert immediately after it
df.insert(col_index + 1, "Log_Detection_Ratio", log_range)

In [5]:
df.head()

,Journal Link,TiC Mxene sensor,Functional Group,Response Time (in min),Current density (in μA/cm2),Detection Range (in M),Log_Detection_Ratio,Compound detected,EN of functional group,Atomic polarizability of functional group (in Å³),Mobility of functional group (in ×10⁻⁸ m²/V·s),Covalent Radii of functional group (in pm),Chemical formula of compound detected,N/O ratio,N/C ratio,C/H ratio,C/O ratio
0,https://www.sciencedirect.com/science/article/...,Graphene/TiC,–O,9.0,1480.000,5.55556*10-17 - 1.11111*10-8,8.301029,Carcinoembryonic antigen(CEA),3.44,0.785,20.64,66.0,C3394H5266N930O1076S15,0.8643,0.2740,0.6445,3.1542
1,https://www.sciencedirect.com/science/article/...,Graphene/TiC,–NH2,9.0,1480.000,5.55556*10-17 - 1.11111*10-8,8.301029,Carcinoembryonic antigen(CEA),3.04,1.096,7.62,71.0,C3394H5266N930O1076S15,0.8643,0.2740,0.6445,3.1542
2,https://www.sciencedirect.com/science/article/...,Graphene/TiC,–F,9.0,1480.000,5.55556*10-17 - 1.11111*10-8,8.301029,Carcinoembryonic antigen(CEA),3.98,0.554,5.74,57.0,C3394H5266N930O1076S15,0.8643,0.2740,0.6445,3.1542
3,https://www.nature.com/articles/s41598-025-267...,PANI/TiC/Au,–NH–,NaN,452.707,7.03482*10-17 - 7.03482*10-14,3.000000,Prostate-specific antigen (PSA),3.04,1.096,7.62,71.0,C1292H2020N352O361S15,0.9750,0.2724,0.6396,3.5789
4,https://www.mdpi.com/2079-6374/14/6/261,GO/TiC/CNT,–O,NaN,1775.700,10-6 - 9770*10-6,3.989895,Hydrogen Peroxide (H2O2),3.44,0.785,20.64,66.0,H2O2,0.0000,NaN,0.0000,0.0000


In [6]:
df = df.dropna(subset=['Functional Group'])

In [7]:
df.shape

(117, 17)

In [8]:
df['TiC Mxene sensor'] = df['TiC Mxene sensor'].str.strip('/')
df['Sensor_parts'] = df['TiC Mxene sensor'].str.split('/')

# Clean spaces
df['Sensor_parts'] = df['Sensor_parts'].apply(
    lambda x: [i.strip() for i in x if i.strip() != '']
)

In [9]:
from itertools import chain

unique_parts = sorted(set(chain.from_iterable(df['Sensor_parts'])))

# Assign codes as strings
component_map = {part: str(i) for i, part in enumerate(unique_parts)}

print(component_map)  

{'Ag': '0', 'AlOOH': '1', 'Apt': '2', 'Au': '3', 'BN': '4', 'CNT': '5', 'COOH': '6', 'Carbon': '7', 'Cd': '8', 'CdS': '9', 'CeMOF': '10', 'Chit': '11', 'Co': '12', 'Co3O4': '13', 'Cu': '14', 'Cu2O': '15', 'CuMOF': '16', 'Fe': '17', 'GO': '18', 'Graphene': '19', 'IL': '20', 'L-Cyst': '21', 'MO': '22', 'Mn': '23', 'MnO2': '24', 'MoS2': '25', 'MoTiC': '26', 'N': '27', 'NH₂': '28', 'NH₂-UiO-66': '29', 'Nafion': '30', 'Ni': '31', 'NiO': '32', 'PANI': '33', 'PDA': '34', 'PEG': '35', 'PPY': '36', 'Pd': '37', 'Pt': '38', 'ReS2': '39', 'S': '40', 'SeS2': '41', 'TB': '42', 'Te': '43', 'TiAlC': '44', 'TiC': '45', 'TiO2': '46', 'UN': '47', 'V2O5': '48', 'aminosilane': '49'}


In [10]:
def encode_sensor(parts):
    codes = [component_map[p] for p in parts]
    return ''.join(sorted(codes))  # ensures consistency

In [11]:
df['Sensor_encoded'] = df['Sensor_parts'].apply(encode_sensor)

In [12]:
df['Sensor_encoded'] = df['Sensor_encoded'].astype(int)

In [13]:
df.drop(columns=['TiC Mxene sensor', 'Sensor_parts'], inplace=True)

In [14]:
df = pd.get_dummies(df, columns=['Functional Group'])

In [15]:
df.head()

,Journal Link,Response Time (in min),Current density (in μA/cm2),Detection Range (in M),Log_Detection_Ratio,Compound detected,EN of functional group,Atomic polarizability of functional group (in Å³),Mobility of functional group (in ×10⁻⁸ m²/V·s),Covalent Radii of functional group (in pm),...,C/H ratio,C/O ratio,Sensor_encoded,Functional Group_–F,Functional Group_–N,Functional Group_–NH2,Functional Group_–NH–,Functional Group_–O,Functional Group_–OH,Functional Group_═O
0,https://www.sciencedirect.com/science/article/...,9.0,1480.000,5.55556*10-17 - 1.11111*10-8,8.301029,Carcinoembryonic antigen(CEA),3.44,0.785,20.64,66.0,...,0.6445,3.1542,1945,False,False,False,False,True,False,False
1,https://www.sciencedirect.com/science/article/...,9.0,1480.000,5.55556*10-17 - 1.11111*10-8,8.301029,Carcinoembryonic antigen(CEA),3.04,1.096,7.62,71.0,...,0.6445,3.1542,1945,False,False,True,False,False,False,False
2,https://www.sciencedirect.com/science/article/...,9.0,1480.000,5.55556*10-17 - 1.11111*10-8,8.301029,Carcinoembryonic antigen(CEA),3.98,0.554,5.74,57.0,...,0.6445,3.1542,1945,True,False,False,False,False,False,False
3,https://www.nature.com/articles/s41598-025-267...,NaN,452.707,7.03482*10-17 - 7.03482*10-14,3.000000,Prostate-specific antigen (PSA),3.04,1.096,7.62,71.0,...,0.6396,3.5789,33345,False,False,False,True,False,False,False
4,https://www.mdpi.com/2079-6374/14/6/261,NaN,1775.700,10-6 - 9770*10-6,3.989895,Hydrogen Peroxide (H2O2),3.44,0.785,20.64,66.0,...,0.0000,0.0000,18455,False,False,False,False,True,False,False


In [16]:
df['Sensor_encoded']

0       1945
1       1945
2       1945
3      33345
4      18455
       ...  
132    36455
133    36455
134       45
135       45
136       45
Name: Sensor_encoded, Length: 117, dtype: int64

In [17]:
df.columns

Index(['Journal Link', 'Response Time (in min)', 'Current density (in μA/cm2)',
       'Detection Range (in M)', 'Log_Detection_Ratio', 'Compound detected',
       'EN of functional group ',
       'Atomic polarizability of functional group (in Å³)',
       'Mobility of functional group (in ×10⁻⁸ m²/V·s)',
       'Covalent Radii of functional group (in pm)',
       'Chemical formula of compound detected', 'N/O ratio', 'N/C ratio',
       'C/H ratio', 'C/O ratio', 'Sensor_encoded', 'Functional Group_–F',
       'Functional Group_–N', 'Functional Group_–NH2', 'Functional Group_–NH–',
       'Functional Group_–O', 'Functional Group_–OH', 'Functional Group_═O'],
      dtype='object')

In [18]:
df = df.drop(columns=[
    'Journal Link',
    'Response Time (in min)',
    'Compound detected',
    'Covalent Radii of functional group (in pm)',
    'N/O ratio',
    'N/C ratio'
])

In [19]:
df.head()

,Current density (in μA/cm2),Detection Range (in M),Log_Detection_Ratio,EN of functional group,Atomic polarizability of functional group (in Å³),Mobility of functional group (in ×10⁻⁸ m²/V·s),Chemical formula of compound detected,C/H ratio,C/O ratio,Sensor_encoded,Functional Group_–F,Functional Group_–N,Functional Group_–NH2,Functional Group_–NH–,Functional Group_–O,Functional Group_–OH,Functional Group_═O
0,1480.000,5.55556*10-17 - 1.11111*10-8,8.301029,3.44,0.785,20.64,C3394H5266N930O1076S15,0.6445,3.1542,1945,False,False,False,False,True,False,False
1,1480.000,5.55556*10-17 - 1.11111*10-8,8.301029,3.04,1.096,7.62,C3394H5266N930O1076S15,0.6445,3.1542,1945,False,False,True,False,False,False,False
2,1480.000,5.55556*10-17 - 1.11111*10-8,8.301029,3.98,0.554,5.74,C3394H5266N930O1076S15,0.6445,3.1542,1945,True,False,False,False,False,False,False
3,452.707,7.03482*10-17 - 7.03482*10-14,3.000000,3.04,1.096,7.62,C1292H2020N352O361S15,0.6396,3.5789,33345,False,False,False,True,False,False,False
4,1775.700,10-6 - 9770*10-6,3.989895,3.44,0.785,20.64,H2O2,0.0000,0.0000,18455,False,False,False,False,True,False,False


In [20]:
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

In [21]:
df.head()

,Current density (in μA/cm2),Detection Range (in M),Log_Detection_Ratio,EN of functional group,Atomic polarizability of functional group (in Å³),Mobility of functional group (in ×10⁻⁸ m²/V·s),Chemical formula of compound detected,C/H ratio,C/O ratio,Sensor_encoded,Functional Group_–F,Functional Group_–N,Functional Group_–NH2,Functional Group_–NH–,Functional Group_–O,Functional Group_–OH,Functional Group_═O
0,1480.000,5.55556*10-17 - 1.11111*10-8,8.301029,3.44,0.785,20.64,C3394H5266N930O1076S15,0.6445,3.1542,1945,0,0,0,0,1,0,0
1,1480.000,5.55556*10-17 - 1.11111*10-8,8.301029,3.04,1.096,7.62,C3394H5266N930O1076S15,0.6445,3.1542,1945,0,0,1,0,0,0,0
2,1480.000,5.55556*10-17 - 1.11111*10-8,8.301029,3.98,0.554,5.74,C3394H5266N930O1076S15,0.6445,3.1542,1945,1,0,0,0,0,0,0
3,452.707,7.03482*10-17 - 7.03482*10-14,3.000000,3.04,1.096,7.62,C1292H2020N352O361S15,0.6396,3.5789,33345,0,0,0,1,0,0,0
4,1775.700,10-6 - 9770*10-6,3.989895,3.44,0.785,20.64,H2O2,0.0000,0.0000,18455,0,0,0,0,1,0,0


In [22]:
# Adding interaction features
# df['EN x C/O'] = df['EN of functional group '] * df['C/O ratio']
# df['Polarizabilty x C/H'] = df['Atomic polarizability of functional group (in Å³)'] * df['C/H ratio']

In [23]:
df.head()

,Current density (in μA/cm2),Detection Range (in M),Log_Detection_Ratio,EN of functional group,Atomic polarizability of functional group (in Å³),Mobility of functional group (in ×10⁻⁸ m²/V·s),Chemical formula of compound detected,C/H ratio,C/O ratio,Sensor_encoded,Functional Group_–F,Functional Group_–N,Functional Group_–NH2,Functional Group_–NH–,Functional Group_–O,Functional Group_–OH,Functional Group_═O
0,1480.000,5.55556*10-17 - 1.11111*10-8,8.301029,3.44,0.785,20.64,C3394H5266N930O1076S15,0.6445,3.1542,1945,0,0,0,0,1,0,0
1,1480.000,5.55556*10-17 - 1.11111*10-8,8.301029,3.04,1.096,7.62,C3394H5266N930O1076S15,0.6445,3.1542,1945,0,0,1,0,0,0,0
2,1480.000,5.55556*10-17 - 1.11111*10-8,8.301029,3.98,0.554,5.74,C3394H5266N930O1076S15,0.6445,3.1542,1945,1,0,0,0,0,0,0
3,452.707,7.03482*10-17 - 7.03482*10-14,3.000000,3.04,1.096,7.62,C1292H2020N352O361S15,0.6396,3.5789,33345,0,0,0,1,0,0,0
4,1775.700,10-6 - 9770*10-6,3.989895,3.44,0.785,20.64,H2O2,0.0000,0.0000,18455,0,0,0,0,1,0,0


In [24]:
# Remove rows where Log_Detection_Ratio is inf or -inf
df = df[~np.isinf(df['Log_Detection_Ratio'])]

# Reset index
df = df.reset_index(drop=True)

In [25]:
# Shift to make all values positive before log
shift_value = abs(df['Current density (in μA/cm2)'].min()) + 1
df['Current density (in μA/cm2)'] = np.log1p(
    df['Current density (in μA/cm2)'] + shift_value
)

In [26]:
df.head()

,Current density (in μA/cm2),Detection Range (in M),Log_Detection_Ratio,EN of functional group,Atomic polarizability of functional group (in Å³),Mobility of functional group (in ×10⁻⁸ m²/V·s),Chemical formula of compound detected,C/H ratio,C/O ratio,Sensor_encoded,Functional Group_–F,Functional Group_–N,Functional Group_–NH2,Functional Group_–NH–,Functional Group_–O,Functional Group_–OH,Functional Group_═O
0,8.920255,5.55556*10-17 - 1.11111*10-8,8.301029,3.44,0.785,20.64,C3394H5266N930O1076S15,0.6445,3.1542,1945,0,0,0,0,1,0,0
1,8.920255,5.55556*10-17 - 1.11111*10-8,8.301029,3.04,1.096,7.62,C3394H5266N930O1076S15,0.6445,3.1542,1945,0,0,1,0,0,0,0
2,8.920255,5.55556*10-17 - 1.11111*10-8,8.301029,3.98,0.554,5.74,C3394H5266N930O1076S15,0.6445,3.1542,1945,1,0,0,0,0,0,0
3,8.772565,7.03482*10-17 - 7.03482*10-14,3.000000,3.04,1.096,7.62,C1292H2020N352O361S15,0.6396,3.5789,33345,0,0,0,1,0,0,0
4,8.959016,10-6 - 9770*10-6,3.989895,3.44,0.785,20.64,H2O2,0.0000,0.0000,18455,0,0,0,0,1,0,0


In [27]:
df = df.drop(columns=['Detection Range (in M)'])

In [28]:
df.head()

,Current density (in μA/cm2),Log_Detection_Ratio,EN of functional group,Atomic polarizability of functional group (in Å³),Mobility of functional group (in ×10⁻⁸ m²/V·s),Chemical formula of compound detected,C/H ratio,C/O ratio,Sensor_encoded,Functional Group_–F,Functional Group_–N,Functional Group_–NH2,Functional Group_–NH–,Functional Group_–O,Functional Group_–OH,Functional Group_═O
0,8.920255,8.301029,3.44,0.785,20.64,C3394H5266N930O1076S15,0.6445,3.1542,1945,0,0,0,0,1,0,0
1,8.920255,8.301029,3.04,1.096,7.62,C3394H5266N930O1076S15,0.6445,3.1542,1945,0,0,1,0,0,0,0
2,8.920255,8.301029,3.98,0.554,5.74,C3394H5266N930O1076S15,0.6445,3.1542,1945,1,0,0,0,0,0,0
3,8.772565,3.000000,3.04,1.096,7.62,C1292H2020N352O361S15,0.6396,3.5789,33345,0,0,0,1,0,0,0
4,8.959016,3.989895,3.44,0.785,20.64,H2O2,0.0000,0.0000,18455,0,0,0,0,1,0,0


In [29]:
df.columns

Index(['Current density (in μA/cm2)', 'Log_Detection_Ratio',
       'EN of functional group ',
       'Atomic polarizability of functional group (in Å³)',
       'Mobility of functional group (in ×10⁻⁸ m²/V·s)',
       'Chemical formula of compound detected', 'C/H ratio', 'C/O ratio',
       'Sensor_encoded', 'Functional Group_–F', 'Functional Group_–N',
       'Functional Group_–NH2', 'Functional Group_–NH–', 'Functional Group_–O',
       'Functional Group_–OH', 'Functional Group_═O'],
      dtype='object')

In [30]:
df = df.drop(columns=['EN of functional group ','Mobility of functional group (in ×10⁻⁸ m²/V·s)'])
df.head()

,Current density (in μA/cm2),Log_Detection_Ratio,Atomic polarizability of functional group (in Å³),Chemical formula of compound detected,C/H ratio,C/O ratio,Sensor_encoded,Functional Group_–F,Functional Group_–N,Functional Group_–NH2,Functional Group_–NH–,Functional Group_–O,Functional Group_–OH,Functional Group_═O
0,8.920255,8.301029,0.785,C3394H5266N930O1076S15,0.6445,3.1542,1945,0,0,0,0,1,0,0
1,8.920255,8.301029,1.096,C3394H5266N930O1076S15,0.6445,3.1542,1945,0,0,1,0,0,0,0
2,8.920255,8.301029,0.554,C3394H5266N930O1076S15,0.6445,3.1542,1945,1,0,0,0,0,0,0
3,8.772565,3.000000,1.096,C1292H2020N352O361S15,0.6396,3.5789,33345,0,0,0,1,0,0,0
4,8.959016,3.989895,0.785,H2O2,0.0000,0.0000,18455,0,0,0,0,1,0,0


In [31]:
df.dtypes

Current density (in μA/cm2)                          float64
Log_Detection_Ratio                                  float64
Atomic polarizability of functional group (in Å³)    float64
Chemical formula of compound detected                 object
C/H ratio                                            float64
C/O ratio                                            float64
Sensor_encoded                                         int64
Functional Group_–F                                    int64
Functional Group_–N                                    int64
Functional Group_–NH2                                  int64
Functional Group_–NH–                                  int64
Functional Group_–O                                    int64
Functional Group_–OH                                   int64
Functional Group_═O                                    int64
dtype: object

In [32]:
# Define output variables
Y = df[['Current density (in μA/cm2)', 'Log_Detection_Ratio']]

# Define input variables
X = df.drop(columns=['Current density (in μA/cm2)',
                     'Log_Detection_Ratio'])

# Keep only numeric inputs
X = X.select_dtypes(include=[np.number])

# Convert everything safely to numeric
X = X.apply(pd.to_numeric, errors='coerce')

# Handle missing values
for col in X.columns:
    X[col] = X[col].fillna(X[col].mean())


df_model = pd.concat([X, Y], axis=1)

print("Rows before cleaning:", len(df_model))

Rows before cleaning: 116


In [33]:
# Drop rows with missing values
df_model_2 = df_model.dropna(subset=[
    'Current density (in μA/cm2)',
    'Log_Detection_Ratio'
])

print("Rows after dropping missing target values:", len(df_model_2))

Rows after dropping missing target values: 113


In [34]:
# Remove Outliers in targets
def remove_outliers_targets(df, target_cols):
    df_clean = df.copy()
    
    for col in target_cols:
        Q1 = df_clean[col].quantile(0.25)
        Q3 = df_clean[col].quantile(0.75)
        IQR = Q3 - Q1
        
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        
        df_clean = df_clean[
            (df_clean[col] >= lower) &
            (df_clean[col] <= upper)
        ]
    
    return df_clean

target_cols_2 = [
    'Current density (in μA/cm2)',
    'Log_Detection_Ratio'
]

In [35]:
df_clean_2 = remove_outliers_targets(df_model_2, target_cols_2)

print("Rows after target-only outlier removal:", len(df_clean_2))

Rows after target-only outlier removal: 95


In [36]:
Y_clean_2 = df_clean_2[['Current density (in μA/cm2)','Log_Detection_Ratio']]

X_clean_2 = df_clean_2.drop(columns=[
    'Current density (in μA/cm2)',
    'Log_Detection_Ratio'
])

In [37]:
df_clean_2.shape

(95, 13)

In [38]:
df_clean_2.columns

Index(['Atomic polarizability of functional group (in Å³)', 'C/H ratio',
       'C/O ratio', 'Sensor_encoded', 'Functional Group_–F',
       'Functional Group_–N', 'Functional Group_–NH2', 'Functional Group_–NH–',
       'Functional Group_–O', 'Functional Group_–OH', 'Functional Group_═O',
       'Current density (in μA/cm2)', 'Log_Detection_Ratio'],
      dtype='object')

In [39]:
from sklearn.feature_selection import VarianceThreshold
selector = VarianceThreshold(threshold=0.01)
X_clean_2 = selector.fit_transform(X_clean_2)

In [40]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor 
from sklearn.neighbors import KNeighborsRegressor
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import r2_score

In [41]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_clean_2, Y_clean_2,
    test_size=0.2,
    random_state=42
)

### GBR

In [42]:
from sklearn.pipeline import Pipeline
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV

# Pipeline
gbr_pipeline = Pipeline([
    ('model', MultiOutputRegressor(
        GradientBoostingRegressor(random_state=42)
    ))
])

# Parameter grid (IMPORTANT: double "__model")
param_grid = {
    'model__estimator__n_estimators': [100, 200, 300],
    'model__estimator__learning_rate': [0.03, 0.05, 0.1],
    'model__estimator__max_depth': [2, 3],
    'model__estimator__min_samples_leaf': [2, 3]
}

# GridSearch
gbr_grid = GridSearchCV(
    gbr_pipeline,
    param_grid,
    cv=5,
    scoring='r2',
    n_jobs=1
)

# Fit
gbr_grid.fit(X_train, y_train)

,estimator,Pipeline(step..._state=42)))])
,param_grid,"{'model__estimator__learning_rate': [0.03, 0.05, ...], 'model__estimator__max_depth': [2, 3], 'model__estimator__min_samples_leaf': [2, 3], 'model__estimator__n_estimators': [100, 200, ...]}"
,scoring,'r2'
,n_jobs,1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,estimator,GradientBoost...ndom_state=42)


In [43]:
print("Best CV R2:", gbr_grid.best_score_)
print("Train R2:", gbr_grid.score(X_train, y_train))
print("Test R2:", gbr_grid.score(X_test, y_test))

Best CV R2: 0.1512113734239489
Train R2: 0.6332633681426675
Test R2: 0.3319752218211302


In [44]:
# # Pipeline
# gbr_pipeline = Pipeline([
#     ('model', MultiOutputRegressor(
#         GradientBoostingRegressor(random_state=42)
#     ))
# ])

# # Parameter grid
# param_grid = {
#     'model__estimator__n_estimators': [150, 200, 300],
#     'model__estimator__learning_rate': [0.03, 0.05, 0.08],
#     'model__estimator__max_depth': [2, 3, 4],
#     'model__estimator__min_samples_leaf': [1, 2, 3]
# }

# # GridSearch
# gbr_grid = GridSearchCV(
#     gbr_pipeline,
#     param_grid,
#     cv=5,
#     scoring='r2',
#     n_jobs=1
# )

# # Fit
# gbr_grid.fit(X_train, y_train)

In [45]:
# print("Best CV R2:", gbr_grid.best_score_)
# print("Train R2:", gbr_grid.score(X_train, y_train))
# print("Test R2:", gbr_grid.score(X_test, y_test))

In [46]:
from sklearn.model_selection import train_test_split

for rs in [0, 10, 20, 42, 100]:
    X_train, X_test, y_train, y_test = train_test_split(
        X_clean_2, Y_clean_2, test_size=0.2, random_state=rs
    )

    gbr_grid.fit(X_train, y_train)
    best_model = gbr_grid.best_estimator_

    print("Random State:", rs)
    print("Best Params:", gbr_grid.best_params_)
    print("Train R2:", best_model.score(X_train, y_train))
    print("Test R2:", best_model.score(X_test, y_test))


Random State: 0
Best Params: {'model__estimator__learning_rate': 0.1, 'model__estimator__max_depth': 3, 'model__estimator__min_samples_leaf': 3, 'model__estimator__n_estimators': 200}
Train R2: 0.9917128119333993
Test R2: 0.20937573141186194
Random State: 10
Best Params: {'model__estimator__learning_rate': 0.03, 'model__estimator__max_depth': 3, 'model__estimator__min_samples_leaf': 2, 'model__estimator__n_estimators': 100}
Train R2: 0.8207928904556212
Test R2: 0.5444159051709869
Random State: 20
Best Params: {'model__estimator__learning_rate': 0.05, 'model__estimator__max_depth': 3, 'model__estimator__min_samples_leaf': 2, 'model__estimator__n_estimators': 300}
Train R2: 0.9929227373113771
Test R2: 0.1908173061191883
Random State: 42
Best Params: {'model__estimator__learning_rate': 0.03, 'model__estimator__max_depth': 2, 'model__estimator__min_samples_leaf': 3, 'model__estimator__n_estimators': 100}
Train R2: 0.6332633681426675
Test R2: 0.3319752218211302
Random State: 100
Best Params

**Best test R2 score: 0.5444159051709869**

**Best Params: {'model__estimator__learning_rate': 0.03, 'model__estimator__max_depth': 3, 'model__estimator__min_samples_leaf': 2, 'model__estimator__n_estimators': 100}
And Random state is set to 10**

**Now let's try adding an interaction feature and lets's check the test R2**

In [47]:
df_clean_2['Polarizabilty x C/H'] = df_clean_2['Atomic polarizability of functional group (in Å³)'] * df_clean_2['C/H ratio']

In [48]:
Y_clean_2 = df_clean_2[['Current density (in μA/cm2)','Log_Detection_Ratio']]

X_clean_2 = df_clean_2.drop(columns=[
    'Current density (in μA/cm2)',
    'Log_Detection_Ratio'
])

In [49]:
for rs in [0, 10, 20, 42, 100]:
    X_train, X_test, y_train, y_test = train_test_split(
        X_clean_2, Y_clean_2, test_size=0.2, random_state=rs
    )

    gbr_grid.fit(X_train, y_train)
    best_model = gbr_grid.best_estimator_

    print("Random State:", rs)
    print("Best Params:", gbr_grid.best_params_)
    print("Train R2:", best_model.score(X_train, y_train))
    print("Test R2:", best_model.score(X_test, y_test))
    print("-"*40)

Random State: 0
Best Params: {'model__estimator__learning_rate': 0.1, 'model__estimator__max_depth': 2, 'model__estimator__min_samples_leaf': 3, 'model__estimator__n_estimators': 100}
Train R2: 0.8780992045329382
Test R2: 0.21899654777014027
----------------------------------------
Random State: 10
Best Params: {'model__estimator__learning_rate': 0.03, 'model__estimator__max_depth': 3, 'model__estimator__min_samples_leaf': 2, 'model__estimator__n_estimators': 100}
Train R2: 0.8265788014985305
Test R2: 0.5422479870649055
----------------------------------------
Random State: 20
Best Params: {'model__estimator__learning_rate': 0.05, 'model__estimator__max_depth': 2, 'model__estimator__min_samples_leaf': 2, 'model__estimator__n_estimators': 300}
Train R2: 0.9436415094783913
Test R2: -0.05761127413422884
----------------------------------------
Random State: 42
Best Params: {'model__estimator__learning_rate': 0.03, 'model__estimator__max_depth': 2, 'model__estimator__min_samples_leaf': 3, 

**Best test R2 score: 0.4961853198794264**

**Best Params: {'model__estimator__learning_rate': 0.05, 'model__estimator__max_depth': 3, 'model__estimator__min_samples_leaf': 2, 'model__estimator__n_estimators': 300}
And Random state is set to 100**

After adding the interaction feature, we observed that the test R2 score has dropped from 0.544 to 0.496

### RF

In [50]:
from sklearn.pipeline import Pipeline
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

# Pipeline
rf_pipeline = Pipeline([
    ('model', MultiOutputRegressor(
        RandomForestRegressor(random_state=42)
    ))
])

# Param grid (NOTE the extra "__estimator__")
param_grid = {
    'model__estimator__n_estimators': [100, 200, 300],
    'model__estimator__max_depth': [4, 5, 6],
    'model__estimator__min_samples_leaf': [1, 2, 3],
    'model__estimator__max_features': ['sqrt', 'log2']
}

# GridSearch
rf_grid = GridSearchCV(
    rf_pipeline,
    param_grid,
    cv=5,
    scoring='r2',
    n_jobs=1
)
  

In [51]:
from sklearn.model_selection import train_test_split

for rs in [0, 10, 20, 42, 100]:
    X_train, X_test, y_train, y_test = train_test_split(
        X_clean_2, Y_clean_2, test_size=0.2, random_state=rs
    )

    rf_grid.fit(X_train, y_train)
    best_model = rf_grid.best_estimator_

    print("Random State:", rs)
    print("Best Params:", rf_grid.best_params_)
    print("Train R2:", best_model.score(X_train, y_train))
    print("Test R2:", best_model.score(X_test, y_test))
    print("-"*40)

Random State: 0
Best Params: {'model__estimator__max_depth': 5, 'model__estimator__max_features': 'sqrt', 'model__estimator__min_samples_leaf': 3, 'model__estimator__n_estimators': 100}
Train R2: 0.4910905750118712
Test R2: 0.06312791204271412
----------------------------------------
Random State: 10
Best Params: {'model__estimator__max_depth': 4, 'model__estimator__max_features': 'sqrt', 'model__estimator__min_samples_leaf': 2, 'model__estimator__n_estimators': 200}
Train R2: 0.4887860315818347
Test R2: 0.2580832326995173
----------------------------------------
Random State: 20
Best Params: {'model__estimator__max_depth': 4, 'model__estimator__max_features': 'sqrt', 'model__estimator__min_samples_leaf': 1, 'model__estimator__n_estimators': 200}
Train R2: 0.5546734010370747
Test R2: -0.10598157111219642
----------------------------------------
Random State: 42
Best Params: {'model__estimator__max_depth': 4, 'model__estimator__max_features': 'sqrt', 'model__estimator__min_samples_leaf'

**Best test R2 score: 0.2580832326995173**

**Best Params: {'model__estimator__max_depth': 4, 'model__estimator__max_features': 'sqrt', 'model__estimator__min_samples_leaf': 2, 'model__estimator__n_estimators': 200}
And Random state is set to 10**

In [52]:
df_clean_2.columns

Index(['Atomic polarizability of functional group (in Å³)', 'C/H ratio',
       'C/O ratio', 'Sensor_encoded', 'Functional Group_–F',
       'Functional Group_–N', 'Functional Group_–NH2', 'Functional Group_–NH–',
       'Functional Group_–O', 'Functional Group_–OH', 'Functional Group_═O',
       'Current density (in μA/cm2)', 'Log_Detection_Ratio',
       'Polarizabilty x C/H'],
      dtype='object')

In [53]:
df_clean_2 = df_clean_2.drop(columns='Polarizabilty x C/H')

In [54]:
Y_clean_2 = df_clean_2[['Current density (in μA/cm2)','Log_Detection_Ratio']]

X_clean_2 = df_clean_2.drop(columns=[
    'Current density (in μA/cm2)',
    'Log_Detection_Ratio'
])

In [55]:
from sklearn.model_selection import train_test_split

for rs in [0, 10, 20, 42, 100]:
    X_train, X_test, y_train, y_test = train_test_split(
        X_clean_2, Y_clean_2, test_size=0.2, random_state=rs
    )

    rf_grid.fit(X_train, y_train)
    best_model = rf_grid.best_estimator_

    print("Random State:", rs)
    print("Best Params:", rf_grid.best_params_)
    print("Train R2:", best_model.score(X_train, y_train))
    print("Test R2:", best_model.score(X_test, y_test))
    print("-"*40)

Random State: 0
Best Params: {'model__estimator__max_depth': 4, 'model__estimator__max_features': 'sqrt', 'model__estimator__min_samples_leaf': 1, 'model__estimator__n_estimators': 100}
Train R2: 0.5688974465003391
Test R2: 0.06435877566153675
----------------------------------------
Random State: 10
Best Params: {'model__estimator__max_depth': 4, 'model__estimator__max_features': 'sqrt', 'model__estimator__min_samples_leaf': 3, 'model__estimator__n_estimators': 100}
Train R2: 0.45612864836943734
Test R2: 0.2380171414043889
----------------------------------------
Random State: 20
Best Params: {'model__estimator__max_depth': 5, 'model__estimator__max_features': 'sqrt', 'model__estimator__min_samples_leaf': 1, 'model__estimator__n_estimators': 100}
Train R2: 0.6774320087447305
Test R2: -0.058780974949611764
----------------------------------------
Random State: 42
Best Params: {'model__estimator__max_depth': 4, 'model__estimator__max_features': 'sqrt', 'model__estimator__min_samples_lea

**Best test R2 score: 0.2791474725268769**

**Best Params: {'model__estimator__max_depth': 6, 'model__estimator__max_features': 'sqrt', 'model__estimator__min_samples_leaf': 1, 'model__estimator__n_estimators': 100}**
**And Random state is set to 100**

Upon removing the interaction feature, we observed that the test R2 score slightly increased from 0.26 to 0.28

The results for the multi output model are as follows:

| Model | Train R² (With Interaction feature) | Test R² (With Interaction feature) | Train R² (Without Interaction feature) | Test R² (Without Interaction feature) |
|------|-----------------------------|----------------------------|--------------------------------|-------------------------------|
| Gradient Boosting Regressor | 0.9959 | 0.4961 | 0.8208 | 0.5444 |
| Random Forest Regressor | 0.4888 | 0.2581 | 0.7507 | 0.2791 |

For the model prediciting both Log_Detection_Ratio and Current_Density together, we observed that adding the Interaction feature (Polarizabilty x C/H) increased the test R2 score